In [ ]:
pip install open_clip_torch

In [ ]:
import torch
from PIL import Image
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Initialize the model, preprocessing function and tokenizer

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16')

### Prepare the Sketch-200 Dataset

In [ ]:
pip install datasets==2.16.0

In [ ]:
from datasets import load_dataset
dataset = load_dataset("songweig/imagenet_sketch")

In [ ]:
print(dataset)

In [ ]:
if sys.modules['imagenet_r_classes']:
  del sys.modules['imagenet_r_classes']

In [ ]:
from imagenet_r_classes import r_class_names, r_wnids, wnid_to_r_index

In [ ]:
import json
from torchvision.datasets.utils import download_url

# Download the official ImageNet class index mapping
download_url(
    "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json",
    "./",
    "imagenet_class_index.json",
)

# Load the JSON mapping file
with open("./imagenet_class_index.json", "r") as f:
    class_idx = json.load(f)

# Convert to a list where index 0-999 corresponds to model outputs
class_names = [class_idx[str(i)][1] for i in range(1000)]

In [ ]:
wnid_to_idx = {class_idx[str(i)][0]: i for i in range(1000)}
idx_to_wnid = {i: class_idx[str(i)][0] for i in range(1000)}

In [ ]:
r_idx_1000 = [wnid_to_idx[w] for w in r_wnids]
idx_to_r_index = {wnid_to_idx[w]: i for w, i in wnid_to_r_index.items()}

In [ ]:
from datasets import ClassLabel

keep = set(r_idx_1000)
sk200 = dataset.filter(lambda y: y in keep, input_columns="label")

In [ ]:
print(sk200)

In [ ]:
new_features = sk200['train'].features.copy()
new_features["label"] = ClassLabel(names=r_class_names)

sk200 = sk200.map(
    lambda y: {"label": idx_to_r_index[y]},
    input_columns="label",
    features=new_features,
)

In [ ]:
sk200['train'][0]

### Prepare the few shot, test and validation features

In [ ]:
from clip_zeroshot import build_and_cache_image_features, build_and_cache_text_features

In [ ]:
import collections
import random

def split_indices(labels, seed, n_cache=16, n_val=10):
    rng = random.Random(seed)
    by_class = collections.defaultdict(list)
    for i, y in enumerate(labels):
        by_class[y].append(i)
    cache, val, test = [], [], []
    for idx in by_class.values():
        idx = idx[:]
        rng.shuffle(idx)
        cache += idx[:n_cache]
        val += idx[n_cache:n_cache + n_val]
        test += idx[n_cache + n_val:]
    return cache, val, test

In [ ]:
class HFImageDataset(Dataset):
    def __init__(self, hf_dataset, preprocess, wnid_to_index):
        self.hf_dataset = hf_dataset
        self.preprocess = preprocess
        self.wnid_to_index = wnid_to_index

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        example = self.hf_dataset[idx]
        image = self.preprocess(example["image"].convert("RGB"))
        label = example["label"]
        return image, label

In [ ]:
full_wrapped = HFImageDataset(sk200['train'], preprocess, wnid_to_r_index)

In [ ]:
full_loader = DataLoader(full_wrapped, batch_size=32)
sk200_all_features = build_and_cache_image_features(model, device, full_loader, './features', 'sk200_all_features')

In [ ]:
cache_idx, val_idx, test_idx = split_indices(sk200['train']['label'], seed=42)
few_shot_img_feats = sk200_all_features['image_features'][cache_idx]
few_shot_labels = sk200_all_features['labels'][cache_idx]
val_feats   = sk200_all_features['image_features'][val_idx]
test_feats  = sk200_all_features['image_features'][test_idx]
test_labels = sk200_all_features['labels'][test_idx]

### Build the text features for zero shot

In [ ]:
if sys.modules['clip_zeroshot']:
  del sys.modules['clip_zeroshot']

In [ ]:
from clip_zeroshot import load_cached_text_features, build_and_cache_image_features, load_cached_image_features

In [ ]:
text_feature_cache = load_cached_text_features('/content/features/r_text-features.pt')
text_features = text_feature_cache['text_features']

### Build the cache model for Tip-Adapter

In [ ]:
import torch.nn.functional as F

one_hot = F.one_hot(few_shot_labels, num_classes=len(r_class_names))

In [ ]:
cache_keys = few_shot_img_feats
cache_values = one_hot.float()

### run zero shot

In [ ]:
if sys.modules['harness']:
  del sys.modules['harness']

In [ ]:
from harness import run_comparison, zero_shot_logits, tip_adapter_logits, ece, accuracy, signed_gap

In [ ]:
metrics = {"accuracy": accuracy, "ece": ece, "signed_gap": signed_gap}

In [ ]:
shared = {
    "test_features": test_feats.to(device),
    "labels": test_labels.to(device),
    "text_features": text_features.to(device),
    "cache_keys": cache_keys.to(device),
    "cache_values": cache_values.to(device),
    "logit_scale": model.logit_scale.exp()
}

In [ ]:
methods = {
    "zero_shot":   {"fn": zero_shot_logits,   "params": {}},
    "tip_adapter": {"fn": tip_adapter_logits, "params": {"alpha": 1.5, "beta": 5.0}}
}

In [ ]:
results = run_comparison(shared, methods, metrics)

In [ ]:
print(results)

In [ ]:
import torch
assert torch.equal(sk200_all_features['labels'], torch.tensor(sk200['train']['label']))

print(len(cache_idx), len(val_idx), len(test_idx))   # 3200, 2000, 4952
assert set(cache_idx).isdisjoint(val_idx) and set(cache_idx).isdisjoint(test_idx) and set(val_idx).isdisjoint(test_idx)
assert len(cache_idx) + len(val_idx) + len(test_idx) == 10152

c = collections.Counter(few_shot_labels.tolist()); print(min(c.values()), max(c.values()))  # 16 16
v = collections.Counter(sk200_all_features['labels'][val_idx].tolist()); print(min(v.values()), max(v.values()))  # 10 10